# Firm-Level Climate Change Exposure — Full Keyword-Discovery Algorithm
**Reproduction of King, Lam & Roberts (2017) discovery loop as adapted by Sautner, van Lent, Vilkov & Zhang (2023, *Journal of Finance*).**

This notebook implements the **full discovery algorithm** (not just applying a ready-made dictionary):
1. Extract text from earnings-call PDFs
2. Split into sentences
3. Define a small set of **initial (seed) climate bigrams**
4. Reference set **R** = sentences containing a seed; Search set **S** = the rest
5. Train a classifier to separate R from S using bigram features (**seeds excluded -> no leakage**)
6. **Discover** new climate bigrams by reading the classifier back (King Step 3)
7. Build the final dictionary **C** = seeds + discovered
8. Score each transcript with **Sautner Eq. (1)** and aggregate to firm-year
9. Validate against the official Sautner dataset

> **Scope note (read this).** Genuine discovery needs a *large, multi-firm* corpus — Sautner used ~800k transcripts across 10,000+ firms. On a handful of transcripts the discovery step is a faithful *demonstration* of the method but will surface noise and a small dictionary. To reproduce the **exact** published numbers, download Sautner's published bigram set from OSF (https://doi.org/10.17605/OSF.IO/FD6JQ) and score full transcripts. Both paths are supported below.


## 0. Setup

In [ ]:
# In Colab, uncomment:
# !pip install pdfplumber nltk scikit-learn pandas tqdm -q
import os, re, glob
from collections import Counter, defaultdict
import numpy as np
import pandas as pd
import nltk
nltk.download('punkt', quiet=True)
try:
    nltk.download('punkt_tab', quiet=True)   # newer nltk
except Exception:
    pass
from nltk.tokenize import sent_tokenize, word_tokenize
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression


## 1. Data extraction  (INPUT -> transcript table)
Point `PDF_FOLDER` at your transcripts. The firm name is parsed **from the transcript header**, not hardcoded — one bug in the original notebook was stamping every file as one firm.
**Output:** DataFrame with `firm, year, quarter, text`.


In [ ]:
PDF_FOLDER = "/content/drive/MyDrive/transcripts"   # <-- change to your folder

def extract_text_from_pdf(path):
    import pdfplumber
    out = []
    with pdfplumber.open(path) as pdf:
        for page in pdf.pages:
            t = page.extract_text()
            if t: out.append(t)
    return "\n".join(out)

def parse_meta(path, text):
    base = os.path.basename(path)
    firm_m = re.search(r'from\s+([A-Z][A-Za-z&\.\- ]+?)\s+\d?Q?\d', text) \
             or re.search(r'^([A-Z][A-Za-z&\.\- ]+?)\s+(?:First|Second|Third|Fourth)\s+Quarter', text, re.M)
    firm = firm_m.group(1).strip() if firm_m else "UNKNOWN"
    q = re.search(r'([1-4])Q|\bQ([1-4])\b|(First|Second|Third|Fourth)\s+Quarter', base + " " + text[:400])
    if q:
        g = q.group(1) or q.group(2) or {"First":"1","Second":"2","Third":"3","Fourth":"4"}[q.group(3)]
        quarter = f"Q{g}"
    else:
        quarter = None
    y = re.search(r'20\d\d', base) or re.search(r'20\d\d', text[:400])
    year = int(y.group()) if y else None
    return firm, year, quarter

def load_transcripts(folder):
    rows = []
    for p in sorted(glob.glob(os.path.join(folder, "*.pdf"))):
        text = extract_text_from_pdf(p)
        firm, year, quarter = parse_meta(p, text)
        rows.append({"firm": firm, "year": year, "quarter": quarter, "text": text})
    return pd.DataFrame(rows)

tx = load_transcripts(PDF_FOLDER)
print("Loaded", len(tx), "transcripts | firms:", tx["firm"].unique())
tx[["firm","year","quarter"]]


## 2. Sentence splitting  (transcript -> sentence table)
Lowercase, collapse whitespace, drop very short fragments. Sentences are the unit the classifier labels.


In [ ]:
sent_rows = []
for _, r in tx.iterrows():
    for s in sent_tokenize(str(r["text"]).lower()):
        s = re.sub(r"\s+", " ", s.replace("\n", " ")).strip()
        if len(s) > 15:
            sent_rows.append({"firm": r["firm"], "year": r["year"],
                              "quarter": r["quarter"], "sentence": s})
sent = pd.DataFrame(sent_rows)
print("Total sentences:", len(sent))
sent.head()


## 3. Initial (seed) climate bigrams
The only human input the algorithm needs: a short list of bigrams that are **unambiguously** about climate change.
> Replace this with Sautner's Table IA.III for a faithful reproduction. The list below is a reasonable stand-in.


In [ ]:
seeds = [
    "climate change","global warming","greenhouse gas","carbon emission","carbon emissions",
    "carbon tax","carbon price","carbon capture","carbon neutral","carbon footprint",
    "net zero","low carbon","clean energy","renewable energy","solar energy","wind energy",
    "energy transition","emission reduction","emissions reduction","fossil fuel",
    "scope 1","scope 2","scope 3","co2 emissions","climate risk","sea level",
]
SEEDS = set(seeds)
print("Seed bigrams:", len(seeds))


## 4. Reference set R and Search set S  (King Step 1, Sautner adaptation)
- **R** (label 1) = sentences containing >=1 seed bigram.
- **S** (label 0) = all other sentences.
The classifier learns from the S sentences that *look like* R.


In [ ]:
def has_seed(s): return int(any(sd in s for sd in SEEDS))
sent["R"] = sent["sentence"].apply(has_seed)
print("Reference sentences R:", int(sent["R"].sum()),
      "| Search set S:", int((sent["R"] == 0).sum()))


## 5. Bigram feature matrix — **seeds excluded**  (fixes label leakage)
The label is defined by seeds, so seeds must not be features. Sautner: features are "bigrams **beyond** the initial set."


In [ ]:
vec = CountVectorizer(ngram_range=(2,2), min_df=3, stop_words="english")
X_all = vec.fit_transform(sent["sentence"])
vocab = np.array(vec.get_feature_names_out())
keep  = np.array([v not in SEEDS for v in vocab])
X, feats = X_all[:, keep], vocab[keep]
print("Candidate (non-seed) bigram features:", X.shape[1])


## 6. Train classifier + discover new bigrams  (King Steps 2-3)
Two discovery scores, faithful to **King's full recipe**:
- **(a) L1-logistic coefficients** — Sautner's approach; L1 also selects a sparse set.
- **(b) King keyword score** — likelihood-style score comparing bigram frequency in R vs S (King Table 1, Step 3c).
`liblinear` + `penalty='l1'` kept for Colab compatibility.


In [ ]:
clf = LogisticRegression(penalty="l1", solver="liblinear",
                         class_weight="balanced", C=0.5, max_iter=3000)
clf.fit(X, sent["R"])
coef = clf.coef_[0]
order = np.argsort(coef)[::-1]
disc_clf = [(feats[i], float(round(coef[i],3))) for i in order if coef[i] > 0]
print("Discovered by classifier (top 25):")
for bg,w in disc_clf[:25]:
    print(f"  {w:>7}  {bg}")


In [ ]:
XR = X[sent["R"].values==1]; XS = X[sent["R"].values==0]
nR, nS = XR.shape[0], XS.shape[0]
dfR = np.asarray((XR>0).sum(axis=0)).ravel()
dfS = np.asarray((XS>0).sum(axis=0)).ravel()
pR = (dfR + 1)/(nR + 2); pS = (dfS + 1)/(nS + 2)
king_score = np.log(pR) - np.log(pS)
order2 = np.argsort(king_score)[::-1]
disc_king = [(feats[i], float(round(king_score[i],3)), int(dfR[i]))
             for i in order2 if dfR[i] >= 2]
print("Discovered by King keyword score (appears in >=2 climate sentences, top 25):")
for bg,sc,d in disc_king[:25]:
    print(f"  {sc:>6}  (R-docs={d:>2})  {bg}")


### Optional practical cleanup
Sautner hand-picks from the top-500 and relies on a huge corpus to drown out noise. On a tiny corpus, apply a light filler stoplist — a *practical add-on*, not the canonical algorithm.


In [ ]:
FILLER = {"ve gone","ve said","moving parts","just quickly","end end","far future",
          "lot good","good returns","great progress","just given","going play",
          "darren wondering","equation ve","play important"}
discovered = [bg for bg,_ in disc_clf if bg not in FILLER]
print("Discovered after cleanup:", len(discovered))


## 7. Final climate dictionary C
`C = seeds ∪ discovered`. Use a **set** (the original used a list with duplicates).


In [ ]:
C = sorted(SEEDS | set(discovered))
Cset = set(C)
print(f"|C| = {len(C)}  ({len(SEEDS)} seed + {len(discovered)} discovered)")


## 8. Score each transcript — **Sautner Eq. (1)** and aggregate to firm-year
$$CCExposure_{i,t}=\frac{1}{B_{i,t}}\sum_{b}^{B_{i,t}}\mathbb{1}[b\in C]$$
Denominator B = total bigrams = (tokens - 1). Fixes the denominator/counting issues in the original.


In [ ]:
def bigrams_of(text):
    toks = word_tokenize(str(text).lower())
    return [" ".join(p) for p in zip(toks, toks[1:])]

def score_transcript(text, Cset):
    bgs = bigrams_of(text)
    B = max(len(bgs), 1)
    counts = Counter(bgs)
    n_climate = sum(counts[b] for b in Cset)
    return n_climate, B, n_climate / B

res = tx.apply(lambda r: pd.Series(score_transcript(r["text"], Cset),
               index=["n_climate","B","CCExposure"]), axis=1)
tx = pd.concat([tx, res], axis=1)
print(tx[["firm","year","quarter","n_climate","B","CCExposure"]].to_string(index=False))
firm_year = tx.groupby(["firm","year"])["CCExposure"].mean().reset_index()
print("\nFirm-year CCExposure:")
print(firm_year.to_string(index=False))


## 9. Validation against the official Sautner dataset
Merge firm-year output to `firmyear_score_*.csv` on `isin`+`year`. Because the tiny-corpus dictionary is far smaller than Sautner's, expect the *same order of magnitude*, not an exact match. For exact reproduction, use Sautner's OSF dictionary (Section 10).


In [ ]:
OFFICIAL_CSV = "firmyear_score_2024Q4_Version_2025_Jul_03.csv"
EXXON_ISIN = "US30231G1022"
off = pd.read_csv(OFFICIAL_CSV)
off_x = off[off["isin"] == EXXON_ISIN][["year","cc_expo_ew","op_expo_ew","rg_expo_ew","ph_expo_ew"]]
cmp = firm_year.merge(off_x, on="year", how="left")
cmp["ratio_official_to_ours"] = cmp["cc_expo_ew"] / cmp["CCExposure"]
print(cmp.to_string(index=False))


## 10. PHASE 2 — improvements (topic / sentiment / risk / TF-IDF / OSF dictionary)
Ready to use when we start improving; kept separate so the core stays legible.
**(a) Exact reproduction:** load Sautner's OSF bigram list into `Cset`, re-run Section 8.
**(b) Topic measures** (`Opp/Reg/Phy`): rerun 3-7 with topic-specific seeds, intersect with `C`, drop cross-topic bigrams (Sautner §II.A).
**(c) Sentiment — Eq. (2), Loughran-McDonald.**  **(d) Risk — Eq. (3).**  **(e) TF-IDF — Eq. (4).**


In [ ]:
LM_POS = set()   # <- load Loughran-McDonald positive words
LM_NEG = set()   # <- load Loughran-McDonald negative words
RISK   = {"risk","risks","uncertainty","uncertain","volatility","exposure","threat"}

def sentence_measures(sent_df, Cset):
    rows = []
    for (firm,year,quarter), g in sent_df.groupby(["firm","year","quarter"]):
        pos=neg=risk=nclim=0; B=0
        for s in g["sentence"]:
            bgs = bigrams_of(s); B += len(bgs)
            words = set(word_tokenize(s))
            hits = sum(1 for b in bgs if b in Cset)
            if hits:
                nclim += hits
                if words & LM_POS: pos  += hits
                if words & LM_NEG: neg  += hits
                if words & RISK:   risk += hits
        B = max(B,1)
        rows.append(dict(firm=firm,year=year,quarter=quarter,
                         CCSentimentPos=pos/B, CCSentimentNeg=neg/B, CCRisk=risk/B))
    return pd.DataFrame(rows)

def tfidf_exposure(tx_df, Cset):
    N = len(tx_df); doc_freq = Counter(); per_doc = []
    for t in tx_df["text"]:
        bset = set(bigrams_of(t)); per_doc.append(bset)
        for b in (bset & Cset): doc_freq[b]+=1
    out=[]
    for t,bset in zip(tx_df["text"], per_doc):
        bgs = bigrams_of(t); B=max(len(bgs),1); cnt=Counter(bgs); s=0.0
        for b in Cset:
            if cnt[b] and doc_freq[b]>0: s += cnt[b]*np.log(N/doc_freq[b])
        out.append(s/B)
    return out


## What to do next
1. Point `PDF_FOLDER` at your real PDFs and run 1-9 (verified on your 8 ExxonMobil transcripts).
2. Exact reproduction: load Sautner's OSF dictionary into `Cset`, re-run 8, validate in 9.
3. Get more firms — discovery only becomes meaningful across many firms.
4. Then Phase 2: topic + sentiment + risk + TF-IDF.
